# Task 1.2: Filter Perturb-Net by adj_pval < 0.05

**Goal:** Apply stricter FDR filtering to the three condition networks (Rest, Stim8hr, Stim48hr).

**Inputs:** 
- `perturbnet_Rest.csv`
- `perturbnet_Stim8hr.csv`
- `perturbnet_Stim48hr.csv`

**Outputs:**
- `perturb_net_tfs_only_{cond}_filtered.csv` — filtered edge lists
- `perturb_net_tf_summary_{cond}_filtered.csv` — per-TF statistics
- `perturb_net_qc_report_adjpval05.csv` — QC summary
- `perturb_net_tf_summary_consolidated_filtered.csv` — consolidated TF stats

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd
import numpy as np
import gc

DRIVE_BASE = '/content/drive/MyDrive/benchmarking_paper/datasets'
DRIVE_PERTURB = os.path.join(DRIVE_BASE, 'perturbnet_outputs')
CONDITIONS = ['Rest', 'Stim8hr', 'Stim48hr']
QC_THRESHOLD = 0.05

print(f'Working directory: {DRIVE_PERTURB}')
print(f'QC threshold: adj_pval < {QC_THRESHOLD}')

## Step 1: Filter Networks by adj_pval < 0.05

In [ ]:
qc_summary_rows = []

for cond in CONDITIONS:
    print(f'\n{"="*70}')
    print(f'Filtering {cond}')
    print(f'{"="*70}')
    
    # Load unfiltered network
    net_path = os.path.join(DRIVE_PERTURB, f'perturbnet_{cond}.csv')
    print(f'Loading: {net_path}')
    net_df = pd.read_csv(net_path)
    print(f'  Loaded: {len(net_df):,} edges across {net_df["source_TF"].nunique()} TFs')
    
    # Apply filter
    net_filtered = net_df[net_df['adj_p_value'] < QC_THRESHOLD].copy()
    n_edges_removed = len(net_df) - len(net_filtered)
    pct_retained = len(net_filtered) / len(net_df) * 100
    
    print(f'After adj_pval < {QC_THRESHOLD}:')
    print(f'  Edges retained:  {len(net_filtered):,} ({pct_retained:.3f}%)')
    print(f'  Edges removed:   {n_edges_removed:,}')
    
    tfs_before = net_df['source_TF'].nunique()
    tfs_after = net_filtered['source_TF'].nunique()
    print(f'  TFs: {tfs_before} → {tfs_after} ({tfs_before - tfs_after} lost)')
    
    # Save filtered network
    filtered_path = os.path.join(DRIVE_PERTURB, f'perturb_net_tfs_only_{cond}_filtered.csv')
    net_filtered.to_csv(filtered_path, index=False)
    print(f'✓ Saved: {filtered_path}')
    
    # Generate TF summary
    tf_summary = net_filtered.groupby('source_TF').agg({
        'target_gene': 'nunique',
        'log2FC': ['count', 'mean', 'std', 'min', 'max'],
        'adj_p_value': ['min', 'mean', 'median'],
    }).reset_index()
    tf_summary.columns = ['source_TF', 'n_target_genes', 'n_edges',
                          'log2FC_mean', 'log2FC_std', 'log2FC_min', 'log2FC_max',
                          'adj_pval_min', 'adj_pval_mean', 'adj_pval_median']
    tf_summary['culture_condition'] = cond
    tf_summary = tf_summary.sort_values('n_edges', ascending=False)
    
    print(f'  TF summary: {len(tf_summary)} TFs')
    print(f'  Top 5 TFs:')
    print(tf_summary[['source_TF', 'n_edges', 'n_target_genes']].head().to_string(index=False))
    
    # Save TF summary
    tf_summary_path = os.path.join(DRIVE_PERTURB, f'perturb_net_tf_summary_{cond}_filtered.csv')
    tf_summary.to_csv(tf_summary_path, index=False)
    print(f'✓ Saved: {tf_summary_path}')
    
    # Collect QC stats
    qc_summary_rows.append({
        'condition': cond,
        'n_edges_unfiltered': len(net_df),
        'n_edges_filtered': len(net_filtered),
        'edges_removed': n_edges_removed,
        'pct_edges_retained': pct_retained,
        'n_tfs_unfiltered': tfs_before,
        'n_tfs_filtered': tfs_after,
        'tfs_lost': tfs_before - tfs_after,
    })
    
    del net_df, net_filtered, tf_summary
    gc.collect()

print(f'\n{"="*70}')
print('Step 1 complete')
print(f'{"="*70}')

## Step 2: Generate QC Report

In [ ]:
qc_report_df = pd.DataFrame(qc_summary_rows)
print('\n=== QC Filtering Summary (adj_pval < 0.05) ===')
print(qc_report_df.to_string(index=False))

qc_report_path = os.path.join(DRIVE_PERTURB, 'perturb_net_qc_report_adjpval05.csv')
qc_report_df.to_csv(qc_report_path, index=False)
print(f'\n✓ Saved: {qc_report_path}')

## Step 3: Consolidated TF Summary

In [ ]:
consolidated_tf_summary = []

for cond in CONDITIONS:
    tf_summary_path = os.path.join(DRIVE_PERTURB, f'perturb_net_tf_summary_{cond}_filtered.csv')
    tf_df = pd.read_csv(tf_summary_path)
    consolidated_tf_summary.append(tf_df)

consolidated_df = pd.concat(consolidated_tf_summary, ignore_index=True)
consolidated_df = consolidated_df.sort_values(['source_TF', 'culture_condition']).reset_index(drop=True)

consolidated_path = os.path.join(DRIVE_PERTURB, 'perturb_net_tf_summary_consolidated_filtered.csv')
consolidated_df.to_csv(consolidated_path, index=False)

print(f'Consolidated TF summary: {len(consolidated_df)} TF-condition pairs')
print(f'\nSample (first 10 rows):')
print(consolidated_df.head(10).to_string(index=False))
print(f'\n✓ Saved: {consolidated_path}')

## Step 4: Cross-Condition TF Analysis

In [ ]:
tf_sets_filtered = {}

for cond in CONDITIONS:
    net_path = os.path.join(DRIVE_PERTURB, f'perturb_net_tfs_only_{cond}_filtered.csv')
    net_df = pd.read_csv(net_path)
    tf_sets_filtered[cond] = set(net_df['source_TF'].unique())
    del net_df

in_all_three_filtered = tf_sets_filtered['Rest'] & tf_sets_filtered['Stim8hr'] & tf_sets_filtered['Stim48hr']
rest_only_filtered = tf_sets_filtered['Rest'] - tf_sets_filtered['Stim8hr'] - tf_sets_filtered['Stim48hr']
stim8_only_filtered = tf_sets_filtered['Stim8hr'] - tf_sets_filtered['Rest'] - tf_sets_filtered['Stim48hr']
stim48_only_filtered = tf_sets_filtered['Stim48hr'] - tf_sets_filtered['Rest'] - tf_sets_filtered['Stim8hr']
act_both_not_rest_filtered = tf_sets_filtered['Stim8hr'] & tf_sets_filtered['Stim48hr'] - tf_sets_filtered['Rest']

print('\n=== TF Presence After adj_pval < 0.05 Filter ===')
print(f'TFs present in ALL 3 conditions:       {len(in_all_three_filtered)}')
print(f'TFs present in Rest only:              {len(rest_only_filtered)}')
print(f'TFs present in Stim8hr only:           {len(stim8_only_filtered)}')
print(f'TFs present in Stim48hr only:          {len(stim48_only_filtered)}')
print(f'TFs in both activation, not rest:      {len(act_both_not_rest_filtered)}')

if len(in_all_three_filtered) > 0:
    print(f'\nExample core TFs (in all 3 conditions, first 10):')
    for tf in sorted(in_all_three_filtered)[:10]:
        print(f'  {tf}')

print(f'\n✓ Task 1.2 complete: all outputs saved to {DRIVE_PERTURB}')